In [2]:
import os
import urllib.request
from tqdm import tqdm
import glob

# Base URL
base_url = "https://data.chc.ucsb.edu/products/CHIRP-v3.0/monthly/africa/tifs"

# Output folder
output_folder = "../data/raw/chirps"
os.makedirs(output_folder, exist_ok=True)

print(f"Downloading to: {os.path.abspath(output_folder)}")


In [3]:
# Build list of all files to download (1981-2025)
files_to_download = []

for year in range(1981, 2026):  # 1981 to 2025
    for month in range(1, 13):
        filename = f"chirp-v3.0.{year}.{month:02d}.tif"
        url = f"{base_url}/{filename}"
        files_to_download.append((filename, url))

print(f"Total files to download: {len(files_to_download)}")
print(f"First file: {files_to_download[0][0]}")
print(f"Last file: {files_to_download[-1][0]}")

Total files to download: 540
First file: chirp-v3.0.1981.01.tif
Last file: chirp-v3.0.2025.12.tif


In [4]:
# Download all files with progress bar
failed = []

for filename, url in tqdm(files_to_download, desc="Downloading CHIRPS"):
    filepath = os.path.join(output_folder, filename)
    
    # Skip if already downloaded
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1000:
        continue
    
    try:
        urllib.request.urlretrieve(url, filepath)
    except Exception as e:
        failed.append((filename, str(e)))
        print(f"\n❌ Failed: {filename} — {e}")

print(f"\n\n{'=' * 50}")
print(f"Download complete!")
print(f"Failed downloads: {len(failed)}")
if failed:
    for name, err in failed[:10]:
        print(f"  - {name}: {err}")


❌ Failed: chirp-v3.0.2025.08.tif — <urlopen error retrieval incomplete: got only 4161285 out of 10454604 bytes>




Download complete!
Failed downloads: 1
  - chirp-v3.0.2025.08.tif: <urlopen error retrieval incomplete: got only 4161285 out of 10454604 bytes>


In [5]:
import os
import urllib.request

# Retry the failed file
base_url = "https://data.chc.ucsb.edu/products/CHIRP-v3.0/monthly/africa/tifs"
output_folder = "../data/raw/chirps"

failed_filename = "chirp-v3.0.2025.08.tif"
failed_url = f"{base_url}/{failed_filename}"
failed_path = os.path.join(output_folder, failed_filename)

# Delete any partial file
if os.path.exists(failed_path):
    os.remove(failed_path)
    print(f"🗑️ Removed partial file")

# Retry download
print(f"Retrying: {failed_filename}")
try:
    urllib.request.urlretrieve(failed_url, failed_path)
    size_mb = os.path.getsize(failed_path) / (1024**2)
    print(f"✅ Downloaded successfully: {size_mb:.2f} MB")
except Exception as e:
    print(f"❌ Failed again: {e}")
    print("Try running this cell again in a few minutes.")

🗑️ Removed partial file
Retrying: chirp-v3.0.2025.08.tif
✅ Downloaded successfully: 9.97 MB


In [6]:
import glob
import os

output_folder = "../data/raw/chirps"
downloaded = glob.glob(os.path.join(output_folder, "chirp-v3.0.*.tif"))

# Check for missing months
expected = []
for year in range(1981, 2026):
    for month in range(1, 13):
        expected.append(f"chirp-v3.0.{year}.{month:02d}.tif")

downloaded_names = [os.path.basename(f) for f in downloaded]
missing = [f for f in expected if f not in downloaded_names]

print("=" * 50)
print("DOWNLOAD VERIFICATION")
print("=" * 50)
print(f"Expected files: {len(expected)}")
print(f"Downloaded files: {len(downloaded)}")
print(f"Missing files: {len(missing)}")

if missing:
    print("\n⚠️ Missing files:")
    for f in missing:
        print(f"  - {f}")
else:
    print("\n🎉 All 540 files downloaded successfully!")

# Total size
total_size = sum(os.path.getsize(f) for f in downloaded) / (1024**3)
print(f"\nTotal size: {total_size:.2f} GB")

DOWNLOAD VERIFICATION
Expected files: 540
Downloaded files: 540
Missing files: 0

🎉 All 540 files downloaded successfully!

Total size: 5.04 GB


In [7]:
import rioxarray
import os

# Test opening a few files from different years
test_files = [
    "../data/raw/chirps/chirp-v3.0.1981.01.tif",  # First file
    "../data/raw/chirps/chirp-v3.0.2000.06.tif",  # Middle
    "../data/raw/chirps/chirp-v3.0.2025.12.tif",  # Last file
]

for test_file in test_files:
    if os.path.exists(test_file):
        data = rioxarray.open_rasterio(test_file)
        print(f"\n{'=' * 50}")
        print(f"File: {os.path.basename(test_file)}")
        print(f"{'=' * 50}")
        print(f"Shape: {data.shape}")
        print(f"CRS: {data.rio.crs}")
        print(f"Bounds: {data.rio.bounds()}")
        print(f"Precipitation range: {float(data.min()):.2f} to {float(data.max()):.2f} mm/month")
        data.close()
    else:
        print(f"❌ File not found: {test_file}")


File: chirp-v3.0.1981.01.tif
Shape: (1, 1600, 1500)
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Bounds: (-20.0, -40.000001192092896, 55.00000111758709, 40.0)
Precipitation range: 0.00 to 2021.13 mm/month

File: chirp-v3.0.2000.06.tif
Shape: (1, 1600, 1500)
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Bounds: (-20.0, -40.000001192092896, 55.00000111758709, 40.0)
Precipitation range: 0.00 to 991.60 mm/month

File: chirp-v3.0.2025.12.tif
Shape: (1, 1600, 1500)
CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","912

In [8]:
import rioxarray

# Load one file
data = rioxarray.open_rasterio("../data/raw/chirps/chirp-v3.0.2000.06.tif")

# Ghana's approximate bounding box
ghana_bounds = {
    'min_lon': -3.5,
    'max_lon': 1.2,
    'min_lat': 4.5,
    'max_lat': 11.2
}

print("=" * 50)
print("GHANA COVERAGE CHECK")
print("=" * 50)
print(f"Data bounds: {data.rio.bounds()}")
print(f"Ghana bounds: {ghana_bounds}")

# Check overlap
data_bounds = data.rio.bounds()  # (minx, miny, maxx, maxy)
if (data_bounds[0] <= ghana_bounds['min_lon'] and
    data_bounds[1] <= ghana_bounds['min_lat'] and
    data_bounds[2] >= ghana_bounds['max_lon'] and
    data_bounds[3] >= ghana_bounds['max_lat']):
    print("\n✅ Ghana is FULLY covered by the data!")
else:
    print("\n⚠️ Partial coverage — check bounds carefully")

data.close()

GHANA COVERAGE CHECK
Data bounds: (-20.0, -40.000001192092896, 55.00000111758709, 40.0)
Ghana bounds: {'min_lon': -3.5, 'max_lon': 1.2, 'min_lat': 4.5, 'max_lat': 11.2}

✅ Ghana is FULLY covered by the data!
